# 04 - Feature Engineering ve Feature Selection

**Sorumlu:** Melike Türkdal

**Ödevdeki karşılığı:** 2. Veri Ön İşleme (Feature Engineering, Feature Selection/Importance)

In [1]:
# KURULUM: bu hucreyi her oturumun basinda bir kez calistir.
# Repoyu Colab'e indirir; boylece src/ortak.py kullanilabilir.
import os
REPO_URL = "https://github.com/zeyneppbaran/ml-proje.git"  # TODO: kendi repo adresiniz
if not os.path.exists("/content/ml-proje"):
    !git clone {REPO_URL} /content/ml-proje
else:
    !git -C /content/ml-proje pull
%cd /content/ml-proje

Cloning into '/content/ml-proje'...
remote: Enumerating objects: 225, done.
remote: Counting objects: 100% (225/225), done.
remote: Compressing objects: 100% (199/199), done.
remote: Total 225 (delta 119), reused 72 (delta 21), pack-reused 0 (from 0)
Receiving objects: 100% (225/225), 4.22 MiB | 10.07 MiB/s, done.
Resolving deltas: 100% (119/119), done.
/content/ml-proje


In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from src.ortak import veri_yukle, veriyi_bol, on_isleyici, ortak_cv, RANDOM_STATE

df = veri_yukle()
df.head()

,age,job,marital,education,default,housing,loan,contact,month,day_of_week,...,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y,daha_once_aranmis,pdays_kayitli
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0,0
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0,0
2,37,services,married,high.school,no,yes,no,telephone,may,mon,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0,0
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0,0
4,56,services,married,high.school,no,no,yes,telephone,may,mon,...,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,0,0,0


## D4. Yeni özellikler

### D4.1 Ortak Pipeline'da Üretilen Özelliklerin Analizi

`veri_yukle()` içinde (Kişi 3) iki özellik türetilmiştir: `daha_once_aranmis = (previous > 0)` ve `pdays_kayitli = (pdays != 999)`. Bu bölümde üç soru ölçülerek cevaplanır:
1. `daha_once_aranmis` yeni bilgi taşıyor mu?
2. `pdays_kayitli` neden gerekli?
3. İkisi modelin performansına katkı sağlıyor mu? (ablasyon testi)

In [4]:
# daha_once_aranmis, poutcome'un 'nonexistent' olmamasıyla birebir aynı mı?
esdeger = (df["daha_once_aranmis"] == (df["poutcome"] != "nonexistent").astype(int)).all()
print("daha_once_aranmis == (poutcome != 'nonexistent') tüm satırlarda:", esdeger)
display(pd.crosstab(df["daha_once_aranmis"], df["poutcome"]))

daha_once_aranmis == (poutcome != 'nonexistent') tüm satırlarda: True


poutcome,failure,nonexistent,success
daha_once_aranmis,,,
0,0,35551,0
1,4252,0,1373


In [5]:
# pdays=999 -> NaN -> medyan ile doldurulunca ne oluyor? (sadece train üzerinde)
X_train, X_val, X_test, y_train, y_val, y_test = veriyi_bol(df)
medyan = X_train["pdays"].median()
tr = X_train.assign(y=y_train.values)

kayitsiz = tr[tr["pdays"].isna()]        # hiç kaydı olmayanlar -> imputer bunlara medyanı yazacak
gercek   = tr[tr["pdays"] == medyan]     # gerçekten 'medyan' gün önce aranmış olanlar

print(f"Imputer'ın dolduracağı değer (train medyanı): {medyan:.0f} gün")
print(f"Kaydı olmayan müşteri     : {len(kayitsiz):>6} kişi | abonelik oranı %{kayitsiz['y'].mean()*100:.1f}")
print(f"Gerçekten {medyan:.0f} gün önce aranan: {len(gercek):>6} kişi | abonelik oranı %{gercek['y'].mean()*100:.1f}")

# pdays_kayitli'nin poutcome'a eklediği tek yeni bilgi: failure grubunun ikiye ayrılması
display(df[df["poutcome"] == "failure"].groupby("pdays_kayitli")["y"]
        .agg(Kayit="count", Abonelik_Orani="mean").round(3))

Imputer'ın dolduracağı değer (train medyanı): 6 gün
Kaydı olmayan müşteri     :  27745 kişi | abonelik oranı %9.2
Gerçekten 6 gün önce aranan:    290 kişi | abonelik oranı %71.7


,Kayit,Abonelik_Orani
pdays_kayitli,,
0,4110,0.129
1,142,0.514


**1. Bu iki sütun neden oluşturuldu?**

Veri setindeki pdays sütunu, müşterinin son aranmasından bu yana kaç gün geçtiğini gösterir. Bu bilgi yoksa sütuna 999 yazılmıştır. Ama 999 değeri iki farklı müşteri tipini aynı kalıba sokuyor:

* daha önce **hiç aranmamış** müşteriler,
* daha önce **aranmış ama gün sayısı kaydedilmemiş** müşteriler.

Bu iki grup birbirinden farklıdır, fakat model pdays'e bakarak onları ayıramaz. Ayrıca 999 gerçek bir gün sayısı değildir. Model onu "çok uzun zaman önce arandı" diye yanlış yorumlayabilir.

Bu sorunu çözmek için iki yardımcı sütun türetildi:

* `daha_once_aranmis`: Müşteri daha önce hiç arandı mı?
* `pdays_kayitli`: Gün sayısı gerçekten kayıtlı mı? Bu sütun bir **eksik değer göstergesidir (missing indicator)**. 999 değerleri boş bırakılıp ortanca değerle doldurulduktan sonra, hangi değerlerin gerçek hangilerinin doldurulmuş olduğunu bu sütun hatırlar.

İkisi birlikte müşterileri üç anlamlı gruba ayırır: **hiç aranmamış, aranmış ama kaydı yok, aranmış ve kaydı var.**


In [6]:
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

df_ham = df.copy()
df_ham["pdays"] = df_ham["pdays"].fillna(999)   # 999 kodunu geri koy (ham hali)

def pre_cikar(cikar):
    p = on_isleyici()
    p.transformers = [(a, t, [c for c in cols if c not in cikar]) for a, t, cols in p.transformers]
    return p

modeller = {
    "LR": lambda: LogisticRegression(max_iter=2000, class_weight="balanced"),
    "RF": lambda: RandomForestClassifier(n_estimators=200, min_samples_leaf=5, class_weight="balanced",
                                         random_state=RANDOM_STATE, n_jobs=-1),
}
senaryolar = [
    ("A: Ham (999 kodu, türetilmiş yok)", df_ham, ["daha_once_aranmis", "pdays_kayitli"]),
    ("B: Mevcut ortak pipeline",           df,     []),
    ("C: B - daha_once_aranmis",           df,     ["daha_once_aranmis"]),
    ("D: B - pdays_kayitli",               df,     ["pdays_kayitli"]),
    ("E: B - ikisi",                       df,     ["daha_once_aranmis", "pdays_kayitli"]),
]
satirlar = []
for ad, d, cikar in senaryolar:
    Xtr, _, _, ytr, _, _ = veriyi_bol(d)
    satir = {"Senaryo": ad}
    for m, f in modeller.items():
        s = cross_val_score(Pipeline([("on_isleme", pre_cikar(cikar)), ("model", f())]),
                            Xtr, ytr, cv=ortak_cv(), scoring="roc_auc", n_jobs=-1)
        satir[f"{m} ROC-AUC"] = f"{s.mean():.4f} ± {s.std():.4f}"
    satirlar.append(satir)
display(pd.DataFrame(satirlar))

,Senaryo,LR ROC-AUC,RF ROC-AUC
0,"A: Ham (999 kodu, türetilmiş yok)",0.7888 ± 0.0106,0.7923 ± 0.0098
1,B: Mevcut ortak pipeline,0.7888 ± 0.0106,0.7926 ± 0.0104
2,C: B - daha_once_aranmis,0.7888 ± 0.0107,0.7925 ± 0.0105
3,D: B - pdays_kayitli,0.7883 ± 0.0109,0.7921 ± 0.0101
4,E: B - ikisi,0.7883 ± 0.0109,0.7919 ± 0.0091


**2. Bilgi taşıdıkları hâlde neden skora yansımadı?**

Çünkü model aynı bilgiyi başka bir yerden zaten alıyor. Önceki kampanyanın sonucunu tutan poutcome sütunu üç değer alır: "önceki kampanya yok", "başarısız", "başarılı". "Önceki kampanya yok" değeri, "hiç aranmamış" demekle aynı şeydir. Yani yeni sütunların yaptığı ayrımın büyük kısmını poutcome zaten yapıyor. Yeni sütunların eklediği tek gerçek bilgi çok küçük bir müşteri grubuna ait, bu yüzden genel skorda fark görünmüyor.

**Not: Küçük müşteri grubu (142 kişi)**  
`daha_once_aranmis` ve `pdays_kayitli` sütunlarının `poutcome`'a eklediği tek yeni bilgi, önceki kampanyası başarısız olan müşterileri ikiye ayırmaktır:
- Gün sayısı kayıtlı olmayanlar: kalabalık bir grup, vadeli mevduat açma oranı düşük.
- Gün sayısı kayıtlı olanlar: yalnızca 142 müşteri (verinin yaklaşık binde 3'ü), vadeli mevduat açma oranı yaklaşık 4 kat daha yüksek.

Bu ayrımı yalnızca `pdays_kayitli` yapabilir. Bilgi gerçek ama grup çok küçük olduğu için tüm veri üzerinden ölçülen ROC-AUC skorunda fark yaratmıyor.

**Sonuç:** Sütunların tasarım mantığı doğru. Veri setindeki gerçek bir sorunu (999 kodunun iki farklı grubu karıştırmasını) çözüyorlar. Performansa katkıları küçük, çünkü aynı bilgi büyük ölçüde zaten vardı. Yine de pipeline'da tutuldular, çünkü işlevleri verinin doğru temsil edilmesi. Özellikle pdays_kayitli olmadan, hiç aranmamış müşteriler modele "birkaç gün önce arandı" gibi görünürdü.

### D4.2 Yeni Özellik Adayları

In [9]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score

def ozellik_ekle(d):
    """Satır bazlı, veriden öğrenmeyen dönüşümler (veri sızıntısı yok)."""
    d = d.copy()
    d["yas_grubu"] = pd.cut(d["age"], bins=[0, 25, 35, 45, 55, 60, 120],
                            labels=["<=25", "26-35", "36-45", "46-55", "56-60", ">60"]).astype(str)
    d["calismiyor"] = d["job"].isin(["student", "retired", "unemployed"]).astype(int)
    d["kampanya_log"] = np.log1p(d["campaign"])
    d["toplam_temas"] = d["campaign"] + d["previous"]
    d["ilk_arama"] = (d["campaign"] == 1).astype(int)
    d["kredi_sayisi"] = (d["housing"] == "yes").astype(int) + (d["loan"] == "yes").astype(int)
    d["dusuk_faiz"] = (d["euribor3m"] < 2).astype(int)
    d["cep_x_dusuk_faiz"] = ((d["contact"] == "cellular") & (d["euribor3m"] < 2)).astype(int)
    return d

df_fe = ozellik_ekle(df)

# Adım 1: Tek başına sinyal (gruplara göre vadeli mevduat açma oranı)
for c in ["yas_grubu", "calismiyor", "ilk_arama", "dusuk_faiz", "cep_x_dusuk_faiz"]:
    display(df_fe.groupby(c)["y"].agg(Kayit="count", Acma_Orani="mean").round(3).T)

yas_grubu,26-35,36-45,46-55,56-60,<=25,>60
Kayit,14844.000,12839.000,8247.000,2672.000,1665.00,909.000
Acma_Orani,0.117,0.085,0.087,0.122,0.21,0.455


calismiyor,0,1
Kayit,37569.000,3607.000
Acma_Orani,0.101,0.236


ilk_arama,0,1
Kayit,23542.000,17634.00
Acma_Orani,0.099,0.13


dusuk_faiz,0,1
Kayit,27682.000,13494.000
Acma_Orani,0.048,0.245


cep_x_dusuk_faiz,0,1
Kayit,29011.000,12165.000
Acma_Orani,0.055,0.251


In [10]:
# Adım 2: Modele katkı (eşleştirilmiş 5 katlı CV, sadece train) — yaklaşık 1 dakika sürer
Xtr_fe, _, _, ytr_fe, _, _ = veriyi_bol(df_fe)

adaylar = {"yas_grubu": "kat", "calismiyor": "num", "kampanya_log": "num", "toplam_temas": "num",
           "ilk_arama": "num", "kredi_sayisi": "num", "dusuk_faiz": "num", "cep_x_dusuk_faiz": "num"}

def on_isleme_ekle(ekle):
    p = on_isleyici()
    kat = [c for c in ekle if adaylar[c] == "kat"]
    num = [c for c in ekle if adaylar[c] == "num"]
    if kat: p.transformers.append(("yeni_kat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), kat))
    if num: p.transformers.append(("yeni_num", "passthrough", num))
    return p

modeller = {
    "LR":  lambda: LogisticRegression(max_iter=3000, class_weight="balanced"),
    "HGB": lambda: HistGradientBoostingClassifier(max_iter=200, learning_rate=0.05, random_state=RANDOM_STATE),
}
def cv_skor(ekle, m):
    p = Pipeline([("on_isleme", on_isleme_ekle(ekle)), ("model", modeller[m]())])
    return cross_val_score(p, Xtr_fe, ytr_fe, cv=ortak_cv(), scoring="roc_auc", n_jobs=-1)

temel = {m: cv_skor([], m) for m in modeller}
print({m: round(s.mean(), 4) for m, s in temel.items()}, "<- ortak pipeline (temel)")

satirlar = []
for c in list(adaylar) + ["HEPSİ"]:
    ekle = list(adaylar) if c == "HEPSİ" else [c]
    satir = {"Özellik": c}
    for m in modeller:
        fark = cv_skor(ekle, m) - temel[m]              # aynı katlarda fark
        satir[f"{m} fark"] = round(fark.mean(), 4)
        satir[f"{m} oynama"] = round(fark.std(), 4)
        satir[f"{m} artan kat"] = f"{(fark > 0).sum()}/5"
        satir[f"{m} karar"] = "EKLE" if (fark.mean() > fark.std() and (fark > 0).sum() >= 4) else "-"
    satirlar.append(satir)
sonuc = pd.DataFrame(satirlar)
display(sonuc)

{'LR': np.float64(0.7888), 'HGB': np.float64(0.7993)} <- ortak pipeline (temel)


,Özellik,LR fark,LR oynama,LR artan kat,LR karar,HGB fark,HGB oynama,HGB artan kat,HGB karar
0,yas_grubu,-0.0002,0.0014,3/5,-,-0.0006,0.0012,2/5,-
1,calismiyor,-0.0000,0.0001,1/5,-,-0.0009,0.0019,1/5,-
2,kampanya_log,0.0008,0.0009,4/5,-,0.0000,0.0000,0/5,-
3,toplam_temas,0.0000,0.0000,3/5,-,-0.0016,0.0014,0/5,-
4,ilk_arama,0.0007,0.0008,3/5,-,0.0000,0.0000,0/5,-
5,kredi_sayisi,-0.0000,0.0001,1/5,-,-0.0002,0.0013,2/5,-
6,dusuk_faiz,-0.0001,0.0001,1/5,-,0.0000,0.0000,0/5,-
7,cep_x_dusuk_faiz,0.0000,0.0001,3/5,-,-0.0021,0.0019,0/5,-
8,HEPSİ,0.0005,0.0022,4/5,-,-0.0013,0.0006,0/5,-


### Yorum: Yeni özelliklerin modele katkısı

#### 1. Amaç
Bu analizde, mevcut sütunlardan türetilen sekiz yeni özelliğin modelin tahmin başarısını gerçekten artırıp artırmadığı ölçülmüştür. Yeni bir özelliğin hedefle ilişkili görünmesi (tek başına sinyal taşıması), onun modele katkı sağlayacağı anlamına gelmez. Model aynı bilgiyi başka sütunlardan zaten alıyor olabilir. Bu yüzden her özellik, modele eklendiğinde skoru değiştirip değiştirmediğine bakılarak değerlendirilmiştir.

#### 2. Yöntem
- **Karşılaştırma noktası (temel):** Hiçbir yeni özellik eklenmeden, ortak pipeline ile elde edilen 5 katlı çapraz doğrulama ROC-AUC skoru (Lojistik Regresyon: 0.7888, HistGradientBoosting: 0.7993).
- **Deneme:** Her özellik modele tek tek eklenmiş ve aynı test tekrarlanmıştır. Son satırda ("HEPSİ") sekiz özellik birlikte eklenmiştir.
- **Eşleştirilmiş karşılaştırma:** Temel model ve özellik eklenmiş model, aynı 5 eğitim/doğrulama bölmesinde (kat) çalıştırılmıştır. Böylece her katta iki skor doğrudan karşılaştırılabilmiş ve farkın veri bölünmesinden değil, eklenen özellikten kaynaklandığı güvence altına alınmıştır.
- **İki farklı model:** Lojistik Regresyon (LR, doğrusal) ve HistGradientBoosting (HGB, ağaç tabanlı) kullanılmıştır. Çünkü bir dönüşüm doğrusal modele yarayıp ağaç modeline yaramayabilir.
- **Veri sızıntısı önlemi:** Tüm ölçümler yalnızca eğitim verisi üzerinde yapılmıştır. Doğrulama ve test setleri bu analizde kullanılmamıştır. Yeni özellikler satır bazlı kurallarla üretildiği için veriden istatistik öğrenmezler.
- **Karar kuralı (test öncesinde belirlenmiştir):** Bir özellik ancak (1) ortalama skor artışı katlar arasındaki dalgalanmadan büyükse ve (2) 5 katın en az 4'ünde skor artıyorsa eklenir. Amaç, şans eseri oluşan küçük iyileşmeleri gerçek katkıdan ayırmaktır.

#### 3. Tablonun okunması
- **fark:** Özellik eklendiğinde ROC-AUC'nin temel skora göre ortalama değişimi. Pozitif değer iyileşme, negatif değer kötüleşme, 0 değişiklik yok anlamına gelir.
- **oynama:** Bu farkın 5 kat arasındaki standart sapması. Fark oynamadan küçükse, gözlenen değişim şans eseri olabilir.
- **artan kat:** 5 katın kaçında skorun arttığı. 5/5 tutarlı bir iyileşme, 0/5 hiç iyileşme olmadığını gösterir.
- **karar:** Karar kuralını geçen özellikler "EKLE", geçemeyenler "-" ile gösterilmiştir.

#### 4. Bulgular
- **Hiçbir özellik karar kuralını geçememiştir.** Tüm farklar, katlar arası doğal dalgalanma düzeyinde kalmıştır.
- **Lojistik Regresyon:** `kampanya_log` (5 katın 4'ünde artış) ve `ilk_arama` (5 katın 3'ünde artış) küçük ama çoğunlukla aynı yönde bir iyileşme göstermiştir. Bu, `campaign` sütununun çarpık dağılımını düzeltmenin doğrusal modele yarayacağı hipotezini destekler. Ancak artış, katlar arası dalgalanmayı aşmadığı için anlamlı kabul edilmemiştir. Diğer özelliklerde fark yaklaşık sıfırdır.
- **HistGradientBoosting:** `kampanya_log`, `ilk_arama` ve `dusuk_faiz` özelliklerinde fark tam olarak sıfırdır. Yani ağaç modeli bu özellikler eklense de eklenmese de birebir aynı sonucu üretmiştir. `toplam_temas`, `cep_x_dusuk_faiz` ve `calismiyor` gibi özellikler ise skoru hafifçe düşürmüştür. Sekiz özellik birlikte eklendiğinde 5 katın hiçbirinde iyileşme görülmemiştir.

#### 5. Bulguların açıklaması
1. **Yeni özellikler yeni bilgi getirmez.** Hepsi modelin zaten gördüğü sütunlardan türetilmiştir (`dusuk_faiz` euribor'dan, `yas_grubu` yaştan, `calismiyor` meslekten). Model aynı bilgiye orijinal sütunlar üzerinden zaten erişebilmektedir.
2. **Ağaç modelleri bu dönüşümleri kendileri yapar.** Karar ağaçları veriyi "euribor 2'den küçük mü?", "yaş 60'tan büyük mü?" gibi eşik sorularıyla böler. Log dönüşümü gibi sıralamayı değiştirmeyen işlemler bu soruların cevabını değiştirmediği için ağaç modeline hiçbir etki yapmaz. Sıfır farkların nedeni budur.
3. **Doğrusal model kısmen faydalanabilir.** Lojistik Regresyon her sütunla doğrusal bir ilişki kurar ve çarpık dağılımları ya da eşik etkilerini kendisi modelleyemez. Bu yüzden küçük iyileşmeler yalnızca LR tarafında görülmüştür. Ancak bu iyileşmeler de anlamlı düzeye ulaşmamıştır.
4. **Gereksiz özellikler modeli bozabilir.** Yeni bilgi taşımayan sütunlar, ağaç modelinin yanlış bölmeler yapma ihtimalini artırır. Tüm özellikler birlikte eklendiğinde HGB skorunun her katta düşmesi bunu göstermektedir.

#### 6. Karar
Yeni özelliklerin hiçbiri ortak pipeline'a eklenmemiştir. Bu sonuç, mevcut sütunlarla ulaşılabilecek bilgi sınırına gelindiğini göstermektedir. Modelin başarısını artırmak için mevcut sütunları yeniden düzenlemek yeterli değildir. Müşterinin geliri ya da bankayla geçmiş ilişkisi gibi yeni bir bilgi kaynağı gerekir, ancak bu veri setinde böyle bilgiler bulunmamaktadır. Görüşme süresi (`duration`) ise arama bitmeden bilinemeyeceği ve veri sızıntısı yaratacağı için modelden çıkarılmıştır.

## D5 Feature Selection

**Amaç:** Vadeli mevduat açma kararını tahmin ederken hangileri gerçekten işe yarıyor, hangileri neredeyse hiç katkı sağlamıyor.

### D5a. yöntemi 1 : Mutual Information

In [ ]:
# kod buraya

**Yorum:** (bu çıktı bize ne söylüyor?)

## D5b. Feature selection yöntemi 2

In [ ]:
# kod buraya

**Yorum:** (bu çıktı bize ne söylüyor?)

## D5c. Sonuçların karşılaştırması

In [ ]:
# kod buraya

**Yorum:** (bu çıktı bize ne söylüyor?)